## Car Racing

Goal: Attempt to collect observations on car racing task

In [2]:
from functools import partial

import gymnasium as gym
import torch

from world_models.models.vae import VAE
from world_models.models.mdn_rnn import MDNRNN
from world_models.models.controller import Controller
from world_models.envs.carracing import encode_observation
from world_models.evaluation.rollout import rollout

In [3]:
torch.manual_seed(0)

# Fresh models for integration checks.
vae = VAE().cpu().eval()

memory = MDNRNN(
    latent_dim=32,
    action_dim=3,
    hidden_dim=256,
    num_components=5,
).cpu().eval()

controller = Controller().cpu().eval()

# Bind this VAE to the encoder expected by rollout.
encoder = partial(encode_observation, vae=vae)

In [4]:
env = gym.make(
    "CarRacing-v3",
    continuous=True,
    domain_randomize=False,
)

try:
    torch.manual_seed(0)

    result = rollout(
        env=env,
        encode_observation=encoder,
        memory=memory,
        controller=controller,
        seed=0,
        max_steps=200,
    )
    print(result)
finally:
    env.close()

{'reward': 36.42633228840104, 'steps': 200}


In [16]:
def preprocess_observation(observation):
    """Emulate the published CarRacing wrapper's legacy preprocessing."""
    frame = np.asarray(observation)

    if frame.shape != (96, 96, 3) or frame.dtype != np.uint8:
        raise ValueError(
            f"Expected raw uint8 (96, 96, 3), "
            f"got {frame.dtype} {frame.shape}"
        )

    # Original wrapper removes the bottom instrument panel.
    cropped = frame[:84].astype(np.float64) / 255.0

    # Emulate scipy.misc.imresize's float-to-byte scaling.
    minimum = cropped.min()
    span = cropped.max() - minimum
    if span == 0:
        span = 1.0

    scaled = (cropped - minimum) * (255.0 / span)
    byte_image = (
        np.clip(scaled, 0.0, 255.0) + 0.5
    ).astype(np.uint8)

    resized = np.array(
        Image.fromarray(byte_image).resize(
            (64, 64),
            resample=Image.Resampling.BILINEAR,
        ),
        dtype=np.uint8,
    )

    # Preserve the original wrapper's final uint8 wraparound explicitly.
    transformed = np.rint(
        (1.0 - resized.astype(np.float64)) * 255.0
    ).astype(np.int64)

    return np.remainder(transformed, 256).astype(np.uint8)

In [14]:
import numpy as np
import torch

from world_models.agent import agent_step
from world_models.envs.carracing import (
    encode_observation,
    preprocess_observation,
)

collection_vae = VAE().cpu().eval()
collection_memory = MDNRNN(
    latent_dim=32,
    action_dim=3,
    hidden_dim=256,
    num_components=5,
).cpu().eval()
collection_controller = Controller().cpu().eval()


@torch.no_grad()
def randomize_collection_models(seed):
    rng = np.random.default_rng(seed)
    scale = float(rng.uniform(0.0, 0.01))

    model_scales = [
        (collection_controller, scale),
        (collection_vae, scale / 10_000.0),
        (collection_memory, scale / 10_000.0),
    ]

    for model, effective_scale in model_scales:
        for parameter in model.parameters():
            values = (
                rng.standard_cauchy(size=tuple(parameter.shape))
                * effective_scale
            )

            parameter.copy_(
                torch.as_tensor(
                    values,
                    dtype=parameter.dtype,
                    device=parameter.device,
                )
            )

    return scale

In [11]:
import numpy as np

from world_models.envs.carracing import preprocess_observation


@torch.no_grad()
def collect_episode(env, seed=0, max_steps=1000):
    scale = randomize_collection_models(seed)
    torch.manual_seed(seed)

    # Historical collection deliberately continued despite done signals.
    base_env = env.unwrapped
    observation, _ = base_env.reset(seed=seed)
    state = None

    observations = [preprocess_observation(observation)]
    actions = []
    rewards = []
    terminated_flags = []
    truncated_flags = []

    for t in range(max_steps):
        z = encode_observation(observation, collection_vae)

        if not torch.isfinite(z).all():
            raise RuntimeError(
                f"Non-finite latent: seed={seed}, step={t}. "
                "Do not save this episode."
            )

        action_tensor, state = agent_step(
            z,
            state,
            collection_memory,
            collection_controller,
        )

        state_is_finite = all(
            torch.isfinite(value).all().item()
            for value in state
        )
        if not torch.isfinite(action_tensor).all() or not state_is_finite:
            raise RuntimeError(
                f"Non-finite action or memory: seed={seed}, step={t}. "
                "Do not save this episode."
            )

        action = action_tensor.squeeze(0).cpu().numpy()

        observation, reward, terminated, truncated, _ = (
            base_env.step(action)
        )

        observations.append(preprocess_observation(observation))
        actions.append(action.copy())
        rewards.append(reward)
        terminated_flags.append(terminated)
        truncated_flags.append(truncated)

    return {
        "observations": np.stack(observations),
        "actions": np.asarray(actions, dtype=np.float32),
        "rewards": np.asarray(rewards, dtype=np.float32),
        # These are raw environment signals; collection continues.
        "terminated": np.asarray(terminated_flags, dtype=bool),
        "truncated": np.asarray(truncated_flags, dtype=bool),
        "collection_cutoff": np.asarray(True),
        "seed": np.asarray(seed, dtype=np.int64),
        "randomization_scale": np.asarray(scale),
    }

In [15]:
env = gym.make(
    "CarRacing-v3",
    continuous=True,
    domain_randomize=False,
)

try:
    episode = collect_episode(env, seed=0)
finally:
    env.close()

print("Observations:", episode["observations"].shape)
print("Actions:", episode["actions"].shape)
print("Randomization scale:", episode["randomization_scale"].item())
print("Total reward:", episode["rewards"].sum())

Observations: (1001, 64, 64, 3)
Actions: (1000, 3)
Randomization scale: 0.006369616873214543
Total reward: -78.05644


In [19]:
import json
from pathlib import Path

import PIL
from PIL import Image

# Works when Jupyter starts in the repo root or notebooks directory.
cwd = Path.cwd().resolve()
repo_root = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / "pyproject.toml").is_file()
        and (path / "src" / "world_models").is_dir()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError("Start Jupyter inside world-models-reproduction.")

from datetime import datetime

run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
dataset_dir = repo_root / "data" / f"carracing_pilot_legacy_v1_{run_id}"
dataset_dir.mkdir(parents=True, exist_ok=False)

manifest = {
    "status": "collecting",
    "environment": "CarRacing-v3",
    "preprocessing": "crop84_scipy_bytescale_pillow_legacy_wrap_v1",
    "policy": "random_networks_scaled_cauchy",
    "vae_rnn_parameter_divisor": 10000,
    "max_steps": 1000,
    "ignore_environment_end_signals": True,
    "observation_layout": "T+1,H,W,C",
    "versions": {
        "gymnasium": gym.__version__,
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pillow": PIL.__version__,
    },
    "episodes": [],
}

manifest_path = dataset_dir / "manifest.json"


def save_manifest():
    manifest_path.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )


save_manifest()

env = gym.make(
    "CarRacing-v3",
    continuous=True,
    domain_randomize=False,
)

try:
    for seed in range(20):
        episode = collect_episode(env, seed=seed, max_steps=1000)

        steps = len(episode["actions"])
        assert len(episode["observations"]) == steps + 1

        filename = f"episode_{seed:04d}.npz"
        np.savez_compressed(dataset_dir / filename, **episode)

        manifest["episodes"].append({
            "file": filename,
            "seed": seed,
            "split": "train" if seed < 16 else "validation",
            "steps": steps,
            "reward": float(episode["rewards"].sum()),
            "randomization_scale": float(
                episode["randomization_scale"].item()
            ),
        })
        save_manifest()

        print(
            f"Episode {seed + 1:02d}/20 | "
            f"steps={steps} | "
            f"reward={episode['rewards'].sum():.2f}",
            flush=True,
        )

    manifest["status"] = "complete"
    save_manifest()
finally:
    env.close()

print(f"Saved dataset to: {dataset_dir}")

Episode 01/20 | steps=1000 | reward=-78.06
Episode 02/20 | steps=1000 | reward=-30.91
Episode 03/20 | steps=1000 | reward=-56580.71
Episode 04/20 | steps=1000 | reward=-68557.60
Episode 05/20 | steps=1000 | reward=1.82
Episode 06/20 | steps=1000 | reward=-39.21
Episode 07/20 | steps=1000 | reward=-64.79
Episode 08/20 | steps=1000 | reward=-52.98
Episode 09/20 | steps=1000 | reward=-28.29
Episode 10/20 | steps=1000 | reward=54.39
Episode 11/20 | steps=1000 | reward=-30.23
Episode 12/20 | steps=1000 | reward=-78739.63
Episode 13/20 | steps=1000 | reward=-68255.80
Episode 14/20 | steps=1000 | reward=92.17
Episode 15/20 | steps=1000 | reward=-58397.53
Episode 16/20 | steps=1000 | reward=-24.59
Episode 17/20 | steps=1000 | reward=-58851.75
Episode 18/20 | steps=1000 | reward=-50.17
Episode 19/20 | steps=1000 | reward=-73004.63
Episode 20/20 | steps=1000 | reward=-45.71
Saved dataset to: /Users/cedric/Repos/world-models-reproduction/data/carracing_pilot_legacy_v1_20260910_185047_641885
